# Spark FP-Growth experiment

Evaluate Spark's FP-Growth implementation on transaction baskets extracted from the read-only SQL Server warehouse.


## Imports


In [1]:
import os
import pyodbc
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv
from pyspark.ml.fpm import FPGrowth
from pyspark.sql import Row, SparkSession
from sqlalchemy.engine import URL
from sqlalchemy import create_engine, text


## Configure the read-only SQL Server source


In [ ]:
# Load environment variables from the project root when available.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

datasets_dir = project_root / "datasets"
env_path = project_root / ".env"
if env_path.exists():
    load_dotenv(env_path)
else:
    load_dotenv()

dw_user = (os.getenv("DATAWAREHOUSE_USER") or "").strip()
dw_password = (os.getenv("DATAWAREHOUSE_PASSWORD") or "").strip()
dw_host = (os.getenv("DATAWAREHOUSE_HOST") or "").strip()
dw_database = (os.getenv("DATAWAREHOUSE_DATABASE") or "").strip()

if not all([dw_user, dw_password, dw_host, dw_database]):
    raise ValueError("Missing one or more DATAWAREHOUSE_* environment variables.")

drivers = pyodbc.drivers()
print("Available ODBC drivers:", drivers)

preferred_driver_names = [
    (os.getenv("SQLSERVER_DRIVER") or "").strip(),
    "ODBC Driver 18 for SQL Server",
    "ODBC Driver 17 for SQL Server",
    "SQL Server",
]

driver = next(
    (
        candidate
        for candidate in preferred_driver_names
        if candidate and any(candidate.lower() in d.lower() for d in drivers)
    ),
    None,
)

if not driver:
    raise RuntimeError(
        f"No supported SQL Server ODBC driver found. Available drivers: {drivers}. "
        "Install Microsoft ODBC Driver 18 for SQL Server and verify the 64-bit ODBC Administrator."
    )

print(f"Using SQL Server driver: {driver}")

connection_string = URL.create(
    drivername="mssql+pyodbc",
    username=dw_user,
    password=dw_password,
    host=dw_host,
    database=dw_database,
    query={
        "driver": driver,
        "Encrypt": "yes",
        "TrustServerCertificate": "yes",
        "LoginTimeout": "30",
    },
)

engine = create_engine(connection_string, pool_pre_ping=True, future=True)

with engine.connect() as conn:
    print("Database connection OK:", conn.execute(text("SELECT 1")).scalar())

query = text(
    """
    SELECT 
        dd.[date] AS TRX_date,
        [StoreCode],
        [BillNo],
        [ItemCode],
        ITEMLONGNAME,
        [Quantity],
        DEPARTMENT,
        CLASS,
        SUBCLASS,
        [UOM_CD],
        [TotalAmt],
        [NetValue],
        [WACValue],
        [CSM_QTY],
        [CONSIGN_FINAL_QTY],
        [POS_FINAL_QTY]
    FROM [DBWH_8555].[dbo].[FactSalesTrxNew] fstn
    INNER JOIN dimdate dd ON dd.datekey = fstn.DateKey
    INNER JOIN DimItem di ON fstn.ItemCode = di.ITMCD
    WHERE dd.[date] BETWEEN :start_date AND :end_date AND StoreCode = '002'
    """)


## Load and clean transactions


In [ ]:
with engine.connect() as conn:
    start_date = '2025-01-01'
    end_date = '2025-01-31'

    df = pd.read_sql(query, conn,  params={'start_date': start_date, 'end_date': end_date})


#df = pd.read_csv('t10_trx_data.csv')
    
df = df[~df['ITEMLONGNAME'].str.contains('pepito bag', case=False, na=False)]
#df['month_year'] = pd.to_datetime(df['month_year'], format='%m-%Y')
#df['rule'] = df['antecedents'] + " → " + df['consequents']

df.info()
df.head(1)

## Build baskets and run Spark FP-Growth


In [ ]:
# 1. Start Spark
spark = SparkSession.builder.appName("FPGrowthExample").getOrCreate()

# 2. Buat list of items per transaksi (sama seperti yang sudah kamu lakukan)
# sebelumnya kamu pakai TransactionEncoder, sekarang cukup pakai groupby di pandas
transactions = df.groupby("BillNo")["ItemCode"].apply(list).tolist()

# 3. Convert ke Spark DataFrame
df_spark = spark.createDataFrame([Row(items=items) for items in transactions])

# 4. Jalankan FP-Growth
fpGrowth = FPGrowth(itemsCol="items", minSupport=0.001, minConfidence=0.3)
model = fpGrowth.fit(df_spark)

# 5. Frequent itemsets
model.freqItemsets.show(truncate=False)

# 6. Association rules
model.associationRules.show(truncate=False)

# 7. Prediksi rekomendasi
model.transform(df_spark).show(truncate=False)